In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
from PIL import Image
import requests
from io import BytesIO
from concurrent.futures import ThreadPoolExecutor
from tqdm.notebook import tqdm
import gc

In [ ]:
CSV_FILE_PATH = "/content/train.csv"
URL_COLUMN_NAME = "image_link"
PRICE_COLUMN_NAME = "price"
EPOCHS = 3
BATCH_SIZE = 128
CHUNK_SIZE = 1024
NUM_THREADS = 32
LEARNING_RATE = 1e-3
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


In [ ]:
weights = models.ResNet18_Weights.DEFAULT
resnet = models.resnet18(weights=weights)
resnet.fc = nn.Linear(resnet.fc.in_features, 1)
model = resnet.to(device)
preprocess = weights.transforms()
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

In [ ]:
 session = requests.Session()
adapter = requests.adapters.HTTPAdapter(pool_connections=NUM_THREADS, pool_maxsize=NUM_THREADS)
session.mount('http://', adapter)
session.mount('https://', adapter)
def fetch_and_preprocess(item):
    idx, url, price = item
    try:
        response = session.get(url, timeout=4)
        response.raise_for_status()
        img = Image.open(BytesIO(response.content)).convert('RGB')

        target_price = np.log1p(float(price))
        return preprocess(img), torch.tensor(target_price, dtype=torch.float32)
    except Exception:
        return None, None
df = pd.read_csv(CSV_FILE_PATH)
df = df[df[URL_COLUMN_NAME].notna() & df[PRICE_COLUMN_NAME].notna() & (df[PRICE_COLUMN_NAME] > 0)].reset_index(drop=True)
items = [(i, row[URL_COLUMN_NAME], row[PRICE_COLUMN_NAME]) for i, row in df.iterrows()]


In [ ]:
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    total_samples = 0

    pbar = tqdm(total=len(items), desc=f"Epoch {epoch+1}/{EPOCHS}", unit="img")

    for chunk_start in range(0, len(items), CHUNK_SIZE):
        chunk_items = items[chunk_start : chunk_start + CHUNK_SIZE]

        with ThreadPoolExecutor(max_workers=NUM_THREADS) as executor:
            chunk_results = list(executor.map(fetch_and_preprocess, chunk_items))

        valid_chunk = [(img, target) for img, target in chunk_results if img is not None]

        if valid_chunk:
            images, targets = zip(*valid_chunk)

            for i in range(0, len(valid_chunk), BATCH_SIZE):
                batch_imgs = torch.stack(images[i : i + BATCH_SIZE]).to(device, non_blocking=True)
                batch_targets = torch.stack(targets[i : i + BATCH_SIZE]).unsqueeze(1).to(device, non_blocking=True)

                optimizer.zero_grad()

                with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
                    predictions = model(batch_imgs)
                    loss = criterion(predictions, batch_targets)

                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

                running_loss += loss.item() * len(batch_imgs)
                total_samples += len(batch_imgs)

        pbar.update(len(chunk_items))

        del chunk_results, valid_chunk
        gc.collect()
        if device.type == 'cuda':
            torch.cuda.empty_cache()

    pbar.close()
    epoch_mse = running_loss / max(1, total_samples)
    print(f"Epoch {epoch+1}/{EPOCHS} Finished | Loss (MSE): {epoch_mse:.4f}")
torch.save(model.state_dict(), "resnet18_fast_price_model.pth")

In [ ]:
def predict_price_fast(url):
    model.eval()
    try:
        response = session.get(url, timeout=5)
        response.raise_for_status()
        img = Image.open(BytesIO(response.content)).convert('RGB')
        input_tensor = preprocess(img).unsqueeze(0).to(device)

        with torch.no_grad():
            with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
                output_log = model(input_tensor).item()
                predicted_price = np.expm1(output_log)

        return round(float(predicted_price), 2)
    except Exception as e:
        print(f"Error predicting price: {e}")
        return None

In [ ]:
sample_url = df[URL_COLUMN_NAME].iloc[0]
print(f"Testing with URL: {sample_url}")

predicted = predict_price_fast(sample_url)
print(f"Predicted Price: {predicted}")

In [ ]:
actual_price = df[df[URL_COLUMN_NAME] == sample_url][PRICE_COLUMN_NAME].values[0]

print(f"Image URL: {sample_url}")
print(f"Predicted Price: ${predicted:.2f}")
print(f"Actual Price:    ${actual_price:.2f}")
print(f"Absolute Error:  ${abs(predicted - actual_price):.2f}")